# DebiasConvex Non-Negative Guardrails

This notebook shows the conservative `method_non_neg="svd"` contract for `DCPanelSolver`. The non-negative mode is an experimental point-estimation heuristic: it emits a warning, returns `std=None`, and marks `inference_valid=False`.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from causaltensor.cauest.DebiasConvex import DCPanelSolver

Create a small positive low-rank panel with a block treatment pattern.

In [2]:
M0 = np.outer(np.linspace(1.0, 2.0, 8), np.linspace(0.5, 1.5, 8))
Z = np.zeros_like(M0)
Z[4:, 4:] = 1
O = M0 + 0.25 * Z

solver = DCPanelSolver(Z=Z, O=O)

Fit the experimental non-negative mode and inspect the guardrail outputs.

In [3]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    res = solver.fit(suggest_r=1, method="non-convex", method_non_neg="svd")

print("warning:", str(caught[0].message))
print("tau:", round(float(res.tau), 4))
print("std:", res.std)
print("inference_valid:", res.inference_valid)
print("non_negative_method:", res.non_negative_method)
print("baseline min:", round(float(np.min(res.baseline)), 6))
print("diagnostics:")
for key in ["baseline_min", "negative_fraction", "rank", "stable_rank", "residual_frobenius_norm"]:
    print(f"  {key}: {res.diagnostics[key]:.6f}")

tau: 0.25
std: None
inference_valid: False
non_negative_method: svd
baseline min: 0.5
diagnostics:
  baseline_min: 0.500000
  negative_fraction: 0.000000
  rank: 1.000000
  stable_rank: 1.000000
  residual_frobenius_norm: 0.000003
